In [13]:
from datasets import load_dataset
dataset_path = "dataset/dataset_curation_agent/curated_dataset.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

dataset = dataset.shuffle(seed=42)

total_len = len(dataset)
train_end = int(0.7 * total_len)
eval_end = int(0.9 * total_len)

# Slice the dataset
train_dataset = dataset.select(range(0, train_end))
eval_dataset  = dataset.select(range(train_end, eval_end))
test_dataset  = dataset.select(range(eval_end, total_len))

# Print lengths to verify
print("Total:", total_len)
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print("Test:", len(test_dataset))

print("Train:","example: ",train_dataset[0],"\n")
print("Eval:", len(eval_dataset),"\n example: ",eval_dataset[0],"\n")
print("Test:", len(test_dataset),"\n example: ",test_dataset[0],"\n")

Total: 989
Train: 692
Eval: 198
Test: 99
Train: example:  {'task': 'Fix the issue in the following Python code.', 'buggy_code': 'def test_constant(self):\n    # If we keep achieving a rate of 100 rows in 0.5 seconds, it should\n    # recommend that we keep there\n    rate = WeightedAverageRate(0.5)\n    assert rate.update(100, 0.5) == 100\n    assert self.update(100, 0.5) == 100\n    assert rate.update(100, 0.5) == 100', 'correct_code': 'def test_constant(self):\n    # If we keep achieving a rate of 100 rows in 0.5 seconds, it should\n    # recommend that we keep there\n    rate = WeightedAverageRate(0.5)\n    assert rate.update(100, 0.5) == 100\n    assert rate.update(100, 0.5) == 100\n    assert rate.update(100, 0.5) == 100', 'unit_test': 'def check(candidate):\n    # Instantiate the WeightedAverageRate class\n    rate = candidate(0.5)\n    \n    # Test cases derived from the assertions in the provided code\n    assert rate.update(100, 0.5) == 100\n    assert rate.update(100, 0.5) ==

In [14]:
EVAL_REFERENCES = [ex["correct_code"] for ex in eval_dataset]
TEST_REFERENCES = [ex["correct_code"] for ex in test_dataset]
print("eval_references:", EVAL_REFERENCES[0],"\n")
print("test_references:", TEST_REFERENCES[0],"\n")

eval_references: def brpoplpush(self, src, dst, timeout=0):
    """
    Pop a value off the tail of ``src``, push it on the head of ``dst``
    and then return it.

    This command blocks until a value is in ``src`` or until ``timeout``
    seconds elapse, whichever is first. A ``timeout`` value of 0 blocks
    forever.
    """
    if timeout is None:
        timeout = 0
    return self.execute_command('BRPOPLPUSH', src, dst, timeout) 

test_references: def test_where_with_between(self):
    t = self.con.table('alltypes')

    what = t.filter([t.a > 0, t.f.between(0, 1)])
    result = to_sql(what)
    expected = """SELECT *
 alltypes
E `a` > 0 AND
  `f` BETWEEN 0 AND 1"""
    assert result == expected 



In [15]:
#Just for my train split
def formatting_prompts_func(examples):
    output_text = []
    for i in range(len(examples["task"])):
        instruction = examples["task"][i]
        input_code = examples["buggy_code"][i]
        response = examples["correct_code"][i]

        if input_code.strip():  # if not empty
            text = f'''Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

        ### Instruction:
        {instruction}

        ### Input:
        {input_code}

        ### Response:
        {response}
        '''

        output_text.append(text)

    return output_text


In [16]:
import torch
cuda_available = torch.cuda.is_available()

if cuda_available:
    device_id = 0  # You can change to 1,2,3 if you want other GPUs
    torch.cuda.set_device(device_id)
    # device = torch.device(f"cuda:{device_id}")
    device = torch.device(f"cuda:{device_id}")
    print(f"🖥️ Using GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
else:
    device = torch.device("cpu")
    print("⚙️ No GPU available, using CPU.")

print(f"Device selected: {device}")

🖥️ Using GPU 0: NVIDIA GeForce RTX 4070 SUPER
Device selected: cuda:0


In [17]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    max_seq_length = 512,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

==((====))==  Unsloth 2025.5.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 2. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

In [19]:
import neptune
import neptune.integrations.optuna as optuna_utils
run = neptune.init_run(
    project="casvi/CodeMedic",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIzMTMzYjhhOC1jYzA1LTQ0YjAtOTJjNi1iY2EzM2VhMDY0OTcifQ=="
)


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/casvi/CodeMedic/e/COD-135


In [20]:
import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
import evaluate
from codebleu import compute_codebleu

# Cargar métricas estándar
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
acc = evaluate.load("accuracy")
code_eval = evaluate.load("code_eval")


# Preprocesar logits
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


# Función principal de métricas para el Trainer
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = labels[:, 1:]
    preds = preds[:, :-1]

    # Handle padding/masks
    mask = labels == -100
    labels[mask] = tokenizer.pad_token_id
    preds[mask] = tokenizer.pad_token_id

    # Decode tokens
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Extract completions from "##Fixed Code:"
    decoded_completions = []
    for pred in decoded_preds:
        parts = pred.split("##Fixed Code:")
        completion = parts[-1].strip() if len(parts) > 1 else pred.strip()
        decoded_completions.append(completion)

    # Standard metrics
    bleu_score = bleu.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    rouge_score = rouge.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    accuracy = acc.compute(predictions=preds[~mask], references=labels[~mask])

    # CodeBLEU
    refs = [[ref] for ref in EVAL_REFERENCES]
    codebleu_scores = compute_codebleu(decoded_completions, refs, lang="python")

    #Pass@k
    #Convert to shape: candidates = [[completion1], [completion2], ...]
    # predictions = [[pred] for pred in decoded_completions]
    # pass_at_k_result, _ = code_eval.compute(
    #     references=EVAL_REFERENCES,
    #     predictions=predictions,
    #     k=[1, 2, 4, 8]
    # )

    return {
        "codebleu": codebleu_scores["codebleu"],
        **bleu_score,
        **rouge_score,
        **accuracy,
    }

In [21]:
# from trl import SFTTrainer, SFTConfig
# import time
# start=time.time()
#
# # SFT Config
# config = SFTConfig(
#     dataset_num_proc = 1,
#     #dataset_text_field="prompt",#Depends on the colum of your data set
#     learning_rate=2e-4,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=1,
#     num_train_epochs=5,
#     report_to="none",
#     logging_steps=100,
#     max_steps=2000,
#     eval_accumulation_steps=100,
# )
# trainer = SFTTrainer(
#     model=model,  # base or PEFT model
#     tokenizer=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     formatting_func=formatting_prompts_func,
#     args=config,
#     warmup_steps = 5,
#     weight_decay = 0.01,
#     compute_metrics = compute_metrics,
#     preprocess_logits_for_metrics=preprocess_logits_for_metrics,
# )
# metrics = trainer.evaluate()
# print("Metrics:",metrics)
# trainer.train()
#
#
# end = time.time()
# length = end - start
#
# hours = int(length // 3600)
# minutes = int((length % 3600) // 60)
# seconds = int(length % 60)
#
# print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")


In [22]:
# metrics = trainer.evaluate()
# print("Metrics:",metrics)


In [23]:
from trl import SFTTrainer, SFTConfig
import time
def objective(trial):
    start=time.time()
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5,5e-4, log=True)
    num_epochs = trial.suggest_int("num_train_epochs", 5, 10)
    batch_size=2
    steps_per_epoch = len(train_dataset) // batch_size
    max_steps = num_epochs * steps_per_epoch


    # SFT Config
    config = SFTConfig(
        dataset_num_proc = 1,
        #output_dir="./outputs",
        dataset_text_field="text",#Depends on the colum of your data set
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=batch_size,
        num_train_epochs=num_epochs,
        report_to="none",
        logging_steps=100,
        max_steps=max_steps,
        eval_accumulation_steps=100
    )

    trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_prompts_func,
    args=config,
    warmup_steps = 5,
    weight_decay = 0.01,
    compute_metrics = compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()
    print("Metrics:",metrics)

    # Log trial info to Neptune
    run[f"trial/{trial.number}/metrics"] = metrics
    run[f"trial/{trial.number}/params"] = {
        "learning_rate": learning_rate,
        "num_epochs": num_epochs
    }
    run["eval/metrics"] = metrics



    end = time.time()
    length = end - start

    hours = int(length // 3600)
    minutes = int((length % 3600) // 60)
    seconds = int(length % 60)

    print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")

    return metrics["eval_loss"]  # Or any other metric


In [24]:
import optuna
start=time.time()
neptune_callback = optuna_utils.NeptuneCallback(run)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5, callbacks=[neptune_callback], show_progress_bar=True)
end = time.time()
length = end - start

hours = int(length // 3600)
minutes = int((length % 3600) // 60)
seconds = int(length % 60)
print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")

[I 2025-05-29 12:25:11,517] A new study created in memory with name: no-name-7ae7f20e-4efb-48aa-83c6-e4fb8c4b42b4


  0%|          | 0/6 [00:00<?, ?it/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/692 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/198 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 692 | Num Epochs = 29 | Total steps = 2,422
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 69,730,304/7,000,000,000 (1.00% trained)


...

Step,Training Loss
100,0.027500
200,0.027200
300,0.027600
400,0.027000
500,0.027500
600,0.026800
700,0.026700
800,0.026800
900,0.026800
1000,0.026600


Metrics: {'eval_loss': 2.180399179458618, 'eval_codebleu': 0.5434859158896849, 'eval_bleu': 0.414988742399164, 'eval_precisions': [0.4778668525618682, 0.4235423734741702, 0.3929699564790117, 0.37289120557883987], 'eval_brevity_penalty': 1.0, 'eval_length_ratio': 1.8265741389189534, 'eval_translation_length': 57380, 'eval_reference_length': 31414, 'eval_rouge1': 0.5270227530323486, 'eval_rouge2': 0.48146503890423775, 'eval_rougeL': 0.48880965934756215, 'eval_rougeLsum': 0.5266737469144913, 'eval_accuracy': 0.7894114741373227, 'eval_runtime': 11.9715, 'eval_samples_per_second': 16.539, 'eval_steps_per_second': 2.088}
It took 0 hours, 42 minutes, and 2 seconds to train the model!
[I 2025-05-29 15:44:06,365] Trial 4 finished with value: 2.180399179458618 and parameters: {'learning_rate': 2.1559744875605574e-05, 'num_train_epochs': 8}. Best is trial 3 with value: 1.9904402494430542.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 692 | Num Epochs = 41 | Total steps = 3,460
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 69,730,304/7,000,000,000 (1.00% trained)


Step,Training Loss
100,0.162600
200,0.276800
300,0.228800
400,0.193600
500,0.170600
600,0.158500
700,0.143300
800,0.128700
900,0.121700
1000,0.109500


Metrics: {'eval_loss': 2.431663990020752, 'eval_codebleu': 0.5418108846814232, 'eval_bleu': 0.3960695491002806, 'eval_precisions': [0.4700554302400811, 0.404765664754084, 0.37023929005335166, 0.3493418146479371], 'eval_brevity_penalty': 1.0, 'eval_length_ratio': 1.8204940472400841, 'eval_translation_length': 57189, 'eval_reference_length': 31414, 'eval_rouge1': 0.5202073544488389, 'eval_rouge2': 0.46837332491518685, 'eval_rougeL': 0.4789632222412101, 'eval_rougeLsum': 0.5200691261950019, 'eval_accuracy': 0.7445134429468633, 'eval_runtime': 10.4074, 'eval_samples_per_second': 19.025, 'eval_steps_per_second': 2.402}
It took 0 hours, 53 minutes, and 5 seconds to train the model!
[I 2025-05-29 16:37:12,656] Trial 5 finished with value: 2.431663990020752 and parameters: {'learning_rate': 0.0004119337339386276, 'num_train_epochs': 10}. Best is trial 3 with value: 1.9904402494430542.
It took 4 hours, 12 minutes, and 1 seconds to train the model!


In [26]:
# Get the best parameters
best_trial = study.best_trial

best_params = best_trial.params
print("best_params: ",best_params)

best_value = best_trial.value
print("Eval loss:", best_value)
run.stop()

best_params:  {'learning_rate': 0.00019680111506887668, 'num_train_epochs': 6}
Eval loss: 1.9904402494430542
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] All 0 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/casvi/CodeMedic/e/COD-135/metadata
